### Ready to Go? Let's Start Your Chemical Discovery with CSU-IR And CSU-IR-Psychoactive Substances-Classifiers!

💡 **Pro Tip:** Want to speed things up? You can use a powerful T4 GPU for free!

Just click on **Runtime** at the top-right of the page, select **Change runtime type**, and choose **T4 GPU** from the *Hardware accelerator* dropdown. Of course, if you prefer not to change anything, the code will run perfectly on the default CPU as well.

<hr style="border: none; border-top: 2px dashed #ccc;">

#### 🚀 Final Pre-Flight Check (Crucial!)

1.  **Check Your Connection Status** at the top-right of the page.
    - Colab often connects automatically to your last-used device. If you don't see `Connecting...`, please click the **`Connect`** button manually.
2.  **Wait for the Green Checkmark** (✓) to appear next to your `RAM` and `Disk` status. This confirms you are **"Connected"**.
3.  When Colab asks for permission to access your files, go ahead and click **"Connect to Google Drive"** to authorize it.
4.  Once connected, Run Cells **"Step-by-Step"**.

<hr style="border: none; border-top: 2px dashed #ccc;">

#### ✨ After running, get ready to see the magic unfold:

Our intelligent pipeline will automatically generate and display five exciting analytical results for you:

*   **1. Internal Stress Test**: See how the CSU-IR model performs when searching for internal samples in databases.
*   **2. External Benchmark**: Witness the model's true retrieval power benchmarked against the official NIST test set.
*   **3. Cutting-Edge Substance Search**: Explore retrieval results for New Psychoactive Substances (NPS) across various libraries.
*   **4. Instant Spectra ID**: The PS-IR-Classifier tells you if a substance is psychoactive directly from its IR spectrum, no complex prep needed.
*   **5. One-Click SMILES Analysis**: Just provide a chemical SMILES, and the PS-SMILES-Classifier will instantly give you its classification.

**Note**: Due to Google Drive's memory limitations, we are only demonstrating retrieval on a smaller library in this Colab environment. For instructions on how to run a full evaluation with all libraries, please refer to the "Testing and Inference" section of the README.md in our GitHub repository.

---

### **Step 1: Setting Up Your Workspace** ⚙️

This code cell is the starting point for your discovery journey. It will automatically handle all the essential preparations, including:

1.  **Connecting to your Google Drive** to save and access data.
2.  **Cloning or updating our GitHub repository** to ensure you're using the latest code.
3.  **Installing all required Python libraries** to power the analysis.
4.  **Downloading all model weights and datasets** from Hugging Face and storing them securely in your Drive.

To execute this task, simply hover over the code cell below and click the **Run** button (▶) on the left.

In [ ]:
# ==============================================================================
#  CSU-IR Project Setup Script for Colab
# ==============================================================================
import os
import shutil
from google.colab import drive
from google.colab import userdata
from huggingface_hub import hf_hub_download # Import at the top

# ==================================================================
# --- 1. Mount Google Drive ---
# ==================================================================
print("📁 Mounting Google Drive...")
drive.mount('/content/drive', force_remount=True) # Use force_remount for robustness
print("✅ Google Drive mounted.")


# ==================================================================
# --- 2. Define All Project Paths ---
# ==================================================================
print("\n--- Defining Project Paths ---")
# Project Root
GDRIVE_REPO_PATH = "/content/drive/MyDrive/colab_repos/CSU-IR"

# Requirements file
REQUIREMENTS_FILE = os.path.join(GDRIVE_REPO_PATH, 'requirements_colab.txt')

# --- Data and Model Destination Folders on Google Drive ---
# These are the folders where downloaded files will be saved.
DEST_LIB_GENERAL_PATH = os.path.join(GDRIVE_REPO_PATH, 'CSU-IR', 'data', 'processed_library', 'General')
DEST_LIB_PS_PATH = os.path.join(GDRIVE_REPO_PATH, 'CSU-IR', 'data', 'processed_library', 'PS')
DEST_MODEL_WEIGHTS_PATH = os.path.join(GDRIVE_REPO_PATH, 'CSU-IR', 'check_points')
DEST_PS_CLASSIFIER_WEIGHTS_PATH = os.path.join(GDRIVE_REPO_PATH, 'PS-Classifier', 'check_points')
print("✅ Paths defined.")


# ==================================================================
# --- 3. Clone or Update the 'CSU-IR' repository ---
# ==================================================================
print("\n--- Cloning or Updating Repository ---")
try:
    GIT_TOKEN = userdata.get('GITHUB_PAT')
except userdata.SecretNotFoundError:
    raise Exception("🛑 Error: Please set your GitHub Personal Access Token in Colab Secrets (secret name: GITHUB_PAT)")

if not os.path.exists(GDRIVE_REPO_PATH):
    print(f"⏳ Cloning 'CSU-IR' to: {GDRIVE_REPO_PATH}")
    !git clone -q https://{GIT_TOKEN}@github.com/Hsqcsu/CSU-IR.git {GDRIVE_REPO_PATH}
else:
    print(f"✅ Repository already exists. Pulling latest changes...")
    !cd {GDRIVE_REPO_PATH} && git pull


# ==================================================================
# --- 4. Install Dependencies ---
# ==================================================================
print("\n--- Installing Dependencies ---")
if os.path.exists(REQUIREMENTS_FILE):
    print("⏳ Installing dependencies from requirements_colab.txt...")
    !pip install -q -r {REQUIREMENTS_FILE}
    print("✅ Dependencies installed.")
else:
    print(f"⚠️ Warning: Could not find requirements file at {REQUIREMENTS_FILE}.")


# ==================================================================
# --- 5. Define Download Helper and File Lists ---
# ==================================================================
print("\n--- Preparing for Download ---")
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except userdata.SecretNotFoundError:
    raise Exception("🛑 Error: Please set your Hugging Face Token in Colab Secrets (secret name: HF_TOKEN)")

# --- Reusable Download Helper Function ---
def download_files_from_hf(repo_id, files_dict, destination_dir, token):
    """
    Downloads a dictionary of files from a Hugging Face repo to a destination directory.
    """
    print(f"\n--- Checking files for: {os.path.basename(destination_dir)} ---")
    os.makedirs(destination_dir, exist_ok=True)

    for hf_path, local_name in files_dict.items():
        target_path = os.path.join(destination_dir, local_name)
        if not os.path.exists(target_path):
            print(f"  -> Downloading '{local_name}'...")
            try:
                downloaded_path = hf_hub_download(repo_id=repo_id, filename=hf_path, token=token)
                shutil.copy(downloaded_path, target_path)
                print(f"  ✅ Downloaded successfully.")
            except Exception as e:
                print(f"  ❌ FAILED to download '{local_name}'. Error: {e}")
        else:
            print(f"  👍 '{local_name}' already exists.")

# --- Define All Files to be Downloaded ---
HF_REPO_ID = "Skylight666/CSU-IR"
PROCESSED_LIBRARY_GENERAL_TO_DOWNLOAD = {
    "Processed_Retrieval_Library/General_Retrieval/library_nist_smiles_features_fp16.gz": "library_nist_smiles_features_fp16.gz",
    "Processed_Retrieval_Library/General_Retrieval/library_nist_filter_by_CHONF_smiles_features_fp16.gz": "library_nist_filter_by_CHONF_smiles_features_fp16.gz",
    "Processed_Retrieval_Library/General_Retrieval/library_4k_smiles_features_fp16.gz": "library_4k_smiles_features_fp16.gz",
    "Processed_Retrieval_Library/General_Retrieval/library_200k_smiles_features_fp16.gz": "library_200k_smiles_features_fp16.gz",
    #"Processed_Retrieval_Library/General_Retrieval/library_1M_smiles_features_fp16.gz": "library_1M_smiles_features_fp16.gz",
    #"Processed_Retrieval_Library/General_Retrieval/library_2M_smiles_features_fp16.gz": "library_2M_smiles_features_fp16.gz",
    "Processed_Retrieval_Library/General_Retrieval/smiles_nist.txt": "smiles_nist.txt",
    "Processed_Retrieval_Library/General_Retrieval/smiles_nist_filtered_by_CHONF.txt": "smiles_nist_filtered_by_CHONF.txt",
    "Processed_Retrieval_Library/General_Retrieval/smiles_4k.txt": "smiles_4k.txt",
    "Processed_Retrieval_Library/General_Retrieval/smiles_200k.txt": "smiles_200k.txt",
    #"Processed_Retrieval_Library/General_Retrieval/smiles_1M.txt": "smiles_1M.txt",
    #"Processed_Retrieval_Library/General_Retrieval/smiles_2M.txt": "smiles_2M.txt",
}
PROCESSED_LIBRARY_PS_TO_DOWNLOAD = {
    "Processed_Retrieval_Library/PS_Retrieval/library_combined_PS_smiles_features_fp16.gz": "library_combined_PS_smiles_features_fp16.gz",
    "Processed_Retrieval_Library/PS_Retrieval/library_Derivative_PS_smiles_features_fp16.gz": "library_Derivative_PS_smiles_features_fp16.gz",
    "Processed_Retrieval_Library/PS_Retrieval/library_Existed_PS_smiles_features_fp16.gz": "library_Existed_PS_smiles_features_fp16.gz",
    "Processed_Retrieval_Library/PS_Retrieval/smiles_combined.txt": "smiles_combined.txt",
    "Processed_Retrieval_Library/PS_Retrieval/smiles_Derivative_PS.txt": "smiles_Derivative_PS.txt",
    "Processed_Retrieval_Library/PS_Retrieval/smiles_Existed_PS.txt": "smiles_Existed_PS.txt",
}
MODEL_WEIGHTS_TO_DOWNLOAD = {
    "Model_weights/CSU-IR/best_ir_model_500-4000.pth": "best_ir_model_500-4000.pth",
    "Model_weights/CSU-IR/best_smiles_model_500-4000.pth": "best_smiles_model_500-4000.pth",
}
PS_Classifier_MODEL_WEIGHTS_TO_DOWNLOAD = {
    "Model_weights/PS_Classifier/best_ir_model_650-4000.pth": "best_ir_model_650-4000.pth",
    "Model_weights/PS_Classifier/best_smiles_model_650-4000.pth": "best_smiles_model_650-4000.pth",
    "Model_weights/PS_Classifier/best_IR_Classifier_model.pth": "best_IR_Classifier_model.pth",
    "Model_weights/PS_Classifier/best_Smiles_Classifier_model.pth": "best_Smiles_Classifier_model.pth",
}
print("✅ Download helper and file lists are ready.")

# ==================================================================
# --- 6. Execute All Downloads ---
# ==================================================================
print("\n--- Executing All Downloads ---")
download_files_from_hf(HF_REPO_ID, PROCESSED_LIBRARY_GENERAL_TO_DOWNLOAD, DEST_LIB_GENERAL_PATH, HF_TOKEN)
download_files_from_hf(HF_REPO_ID, PROCESSED_LIBRARY_PS_TO_DOWNLOAD, DEST_LIB_PS_PATH, HF_TOKEN)
download_files_from_hf(HF_REPO_ID, MODEL_WEIGHTS_TO_DOWNLOAD, DEST_MODEL_WEIGHTS_PATH, HF_TOKEN)
download_files_from_hf(HF_REPO_ID, PS_Classifier_MODEL_WEIGHTS_TO_DOWNLOAD, DEST_PS_CLASSIFIER_WEIGHTS_PATH, HF_TOKEN)

# ==================================================================
# --- Finalization ---
# ==================================================================
print("\n\n🎉🎉🎉 Full project setup is complete! 🎉🎉🎉")
print("You can now run your analysis and training notebooks.")

---
### **Step 2: Waking Up the "Brains"** 🧠
*Global Initialization: Load All Models & Data*

Everything is now in place! In this step, we'll "wake up" the core intelligence of the project. This cell performs a one-time loading process, bringing all the necessary models and pre-processed feature libraries into memory. This might take a moment, as it's preparing the entire system for high-speed execution in the tasks 온도 follow.

**Once this is complete, all analysis tools will be ready for action!**

In [ ]:
#@title 🧠 2. Global Initialization (Load All Models & Data)
import sys
import os
import gzip
import torch
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader
from rdkit.Chem import Draw, MolFromSmiles
from google.colab import files
import pandas as pd
import jcamp
import numpy as np
from PIL import Image
import io

# --- 1. Set up project paths ---
print("--- Setting up project paths for module imports ---")
GDRIVE_REPO_PATH = "/content/drive/MyDrive/colab_repos/CSU-IR"
CSU_IR_MODULE_PATH = os.path.join(GDRIVE_REPO_PATH, 'CSU-IR')
PS_Classifier_PATH = os.path.join(GDRIVE_REPO_PATH, 'PS_Classifier')
PS_CLASSIFIER_MODULE_PATH = os.path.join(GDRIVE_REPO_PATH, 'PS-Classifier')
if CSU_IR_MODULE_PATH not in sys.path: sys.path.insert(0, CSU_IR_MODULE_PATH)
if PS_CLASSIFIER_MODULE_PATH not in sys.path: sys.path.insert(0, PS_CLASSIFIER_MODULE_PATH)
print("✅ Project paths set.")

# --- 2. Import custom modules ---
print("\n--- Importing custom modules ---")
from model.IR_encoder import IRModel
from model.SMILES_encoder import SmilesModel
from model.Classifier_model import classifymodel
from data_process.ir_process import preprocess_spectra_higer_500, preprocess_spectra_lower_500
from test_and_infer.infer import get_topK_result
from test_and_infer.infer_SMILES_Classifier import normalize_smiles
print("✅ Custom modules imported.")

# --- 3. Define all file paths ---
print("\n--- Defining all file paths ---")
TOKENIZER_PATH = os.path.join(CSU_IR_MODULE_PATH, "model", "tokenizer-smiles-roberta-1e_new")
PRETRAIN_SMILES_MODEL_PATH = os.path.join(DEST_MODEL_WEIGHTS_PATH, "best_smiles_model_500-4000.pth")
PRETRAIN_IR_MODEL_PATH = os.path.join(DEST_MODEL_WEIGHTS_PATH, "best_ir_model_500-4000.pth")
PS_SMILES_ENCODER_PATH = os.path.join(DEST_PS_CLASSIFIER_WEIGHTS_PATH, "best_smiles_model_650-4000.pth")
PS_IR_ENCODER_PATH = os.path.join(DEST_PS_CLASSIFIER_WEIGHTS_PATH, "best_ir_model_650-4000.pth")
PS_SMILES_CLASSIFIER_PATH = os.path.join(DEST_PS_CLASSIFIER_WEIGHTS_PATH, "best_Smiles_Classifier_model.pth")
PS_IR_CLASSIFIER_PATH = os.path.join(DEST_PS_CLASSIFIER_WEIGHTS_PATH, "best_IR_Classifier_model.pth")
TEST_DATA_PATH = os.path.join(CSU_IR_MODULE_PATH, "data", "test_data")
PS_TEST_DATA_PATH = os.path.join(PS_Classifier_PATH, "data", "test_data")
print("✅ File paths defined.")

# --- 4. Initialize all models and load weights ---
print("\n--- Initializing and loading all models ---")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Retrieval Models
ir_retrieval_model = IRModel()
smiles_retrieval_model = SmilesModel(roberta_model_path=None, roberta_tokenizer_path=TOKENIZER_PATH, feature_dim=768)
ir_retrieval_model.load_state_dict(torch.load(PRETRAIN_IR_MODEL_PATH, map_location=device))
smiles_retrieval_model.load_state_dict(torch.load(PRETRAIN_SMILES_MODEL_PATH, map_location=device))
ir_retrieval_model.to(device).eval()
smiles_retrieval_model.to(device).eval()
print("✅ Retrieval models loaded.")

# Classifier Models
# Note: The encoders for classifiers are separate instances
ir_classifier_encoder = IRModel()
smiles_classifier_encoder = SmilesModel(roberta_model_path=None, roberta_tokenizer_path=TOKENIZER_PATH, feature_dim=768)
ir_classifier_model = classifymodel(dim=1024, num_classes=2)
smiles_classifier_model = classifymodel(dim=1024, num_classes=2)

ir_classifier_encoder.load_state_dict(torch.load(PS_IR_ENCODER_PATH, map_location=device))
smiles_classifier_encoder.load_state_dict(torch.load(PS_SMILES_ENCODER_PATH, map_location=device))
ir_classifier_model.load_state_dict(torch.load(PS_IR_CLASSIFIER_PATH, map_location=device))
smiles_classifier_model.load_state_dict(torch.load(PS_SMILES_CLASSIFIER_PATH, map_location=device))

ir_classifier_encoder.to(device).eval()
smiles_classifier_encoder.to(device).eval()
ir_classifier_model.to(device).eval()
smiles_classifier_model.to(device).eval()
print("✅ Classifier models loaded.")

# --- 5. Define helper functions ---
def load_gz_features(path):
    with gzip.open(path, 'rb') as f: return torch.load(f).to(torch.float32)
def load_smiles_list(path):
    with open(path, 'r', encoding='utf-8') as f: return [line.strip() for line in f if line.strip()]
def draw_molecules(smiles_list, scores):
    mols = [MolFromSmiles(s) for s in smiles_list]
    legends = [f"Score: {s:.4f}" for s in scores] if scores else None
    return Draw.MolsToGridImage([m for m in mols if m], molsPerRow=5, subImgSize=(200, 200), legends=legends)
print("✅ Helper functions defined.")

# --- 6. Load all feature libraries and SMILES lists into memory ---
print("\n--- Loading all feature libraries (this may take a moment) ---")
# General Libraries
library_nist_features = load_gz_features(os.path.join(DEST_LIB_GENERAL_PATH, 'library_nist_smiles_features_fp16.gz'))
library_nist_filter_by_CHONF_features = load_gz_features(os.path.join(DEST_LIB_GENERAL_PATH, 'library_nist_filter_by_CHONF_smiles_features_fp16.gz'))
library_4k_features = load_gz_features(os.path.join(DEST_LIB_GENERAL_PATH, 'library_4k_smiles_features_fp16.gz'))
library_200k_features = load_gz_features(os.path.join(DEST_LIB_GENERAL_PATH, 'library_200k_smiles_features_fp16.gz'))
#library_1M_features = load_gz_features(os.path.join(DEST_LIB_GENERAL_PATH, 'library_1M_smiles_features_fp16.gz'))
#library_2M_features = load_gz_features(os.path.join(DEST_LIB_GENERAL_PATH, 'library_2M_smiles_features_fp16.gz'))

general_smiles_nist = load_smiles_list(os.path.join(DEST_LIB_GENERAL_PATH, 'smiles_nist.txt'))
general_smiles_nist_filter_by_CHONF = load_smiles_list(os.path.join(DEST_LIB_GENERAL_PATH, 'smiles_nist_filtered_by_CHONF.txt'))
general_smiles_4k = load_smiles_list(os.path.join(DEST_LIB_GENERAL_PATH, 'smiles_4k.txt'))
general_smiles_200k = load_smiles_list(os.path.join(DEST_LIB_GENERAL_PATH, 'smiles_200k.txt'))
#general_smiles_1M = load_smiles_list(os.path.join(DEST_LIB_GENERAL_PATH, 'smiles_1M.txt'))
#general_smiles_2M = load_smiles_list(os.path.join(DEST_LIB_GENERAL_PATH, 'smiles_2M.txt'))

# PS Libraries
library_existed_ps_features = load_gz_features(os.path.join(DEST_LIB_PS_PATH, 'library_Existed_PS_smiles_features_fp16.gz'))
library_derivative_ps_features = load_gz_features(os.path.join(DEST_LIB_PS_PATH, 'library_Derivative_PS_smiles_features_fp16.gz'))
library_combined_ps_features = load_gz_features(os.path.join(DEST_LIB_PS_PATH, 'library_combined_PS_smiles_features_fp16.gz'))

ps_smiles_existed = load_smiles_list(os.path.join(DEST_LIB_PS_PATH, 'smiles_Existed_PS.txt'))
ps_smiles_derivative = load_smiles_list(os.path.join(DEST_LIB_PS_PATH, 'smiles_Derivative_PS.txt'))
ps_smiles_combined = load_smiles_list(os.path.join(DEST_LIB_PS_PATH, 'smiles_combined.txt'))
print("✅ All libraries loaded.")

print("\n\n🎉🎉🎉 Global initialization complete! System is ready! 🎉🎉🎉")

In [ ]:
#@title 🔬 Task 1: Internal Test Set Retrieval
#@markdown This test evaluates the model's performance by searching for samples from our internal test set against various general-purpose libraries.
#@markdown ---
#@markdown ### Select a library to search against:
library_to_search = "General 200k Library" #@param ["General 4k Library", "General 200k Library", "General 1M Library", "General 2M Library"]

# --- Map user selection to actual data ---
library_map = {
    "General 4k Library": (library_4k_features, general_smiles_4k),
    "General 200k Library": (library_200k_features, general_smiles_200k),
    #"General 1M Library": (library_1M_features, general_smiles_1M),
    #"General 2M Library": (library_2M_features, general_smiles_2M)
}
selected_features, selected_smiles = library_map[library_to_search]

class IRSmilesDataset(Dataset):
    def __init__(self, ir_spectra, smiles):
        self.ir_spectra = ir_spectra
        self.smiles = smiles

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        return self.ir_spectra[idx], self.smiles[idx]

# --- Define evaluation function ---
def evaluate_retrieval(loader, features_db, smiles_db, name):
    top_1, top_5, top_10, total = 0, 0, 0, 0
    with torch.no_grad():
        for ir_batch, smiles_batch in tqdm(loader, desc=f"Evaluating {name}"):
            ir_features = ir_retrieval_model(ir_batch.to(device))
            for i, ir_feature in enumerate(ir_features):
                original_smiles = normalize_smiles(smiles_batch[i])
                indices, _ = get_topK_result(ir_feature.unsqueeze(0), features_db, 10)
                for rank, lib_idx in enumerate(indices[0]):
                    if lib_idx < len(smiles_db) and original_smiles == normalize_smiles(smiles_db[lib_idx]):
                        if rank == 0: top_1 += 1
                        if rank < 5: top_5 += 1
                        if rank < 10: top_10 += 1
                        break
                total += 1
    print(f"\n--- Results for {name} ---\nTop-1: {top_1/total:.4f} | Top-5: {top_5/total:.4f} | Top-10: {top_10/total:.4f} ({top_1}/{total})")

# --- Load internal test data and run ---
Internal_test_smiles, Internal_test_ir = load_smiles_list(os.path.join(TEST_DATA_PATH, "Internal_test", "Internal_test_smiles.txt")), torch.load(os.path.join(TEST_DATA_PATH, "Internal_test", "Internal_test_ir.pt"))
Internal_test_loader = DataLoader(IRSmilesDataset(Internal_test_ir, Internal_test_smiles), batch_size=208, shuffle=False)
evaluate_retrieval(Internal_test_loader, selected_features, selected_smiles, f"Internal Test in {library_to_search}")

In [ ]:
#@title 🌍 Task 2: External NIST Test Set Retrieval
#@markdown This test benchmarks the model against the official NIST test set, searching in various general-purpose libraries.
#@markdown ---
#@markdown ### Select a library to search against:
library_to_search_nist = "General NIST (CHONF only) Library" #@param ["General NIST Library", "General NIST (CHONF only) Library", "General 4k Library", "General 200k Library", "General 1M Library", "General 2M Library"]

# --- Map user selection to actual data ---
library_map_nist = {
    "General NIST Library": (library_nist_features, general_smiles_nist),
    #"General NIST (CHONF only) Library": (library_nist_filter_by_CHONF_features, general_smiles_nist_filter_by_CHONF),
    "General 4k Library": (library_4k_features, general_smiles_4k),
    "General 200k Library": (library_200k_features, general_smiles_200k),
    #"General 1M Library": (library_1M_features, general_smiles_1M),
    #"General 2M Library": (library_2M_features, general_smiles_2M)
}
selected_features_nist, selected_smiles_nist = library_map_nist[library_to_search_nist]

# --- Load NIST test data and run ---
External_nist_test_smiles, External_nist_test_ir = load_smiles_list(os.path.join(TEST_DATA_PATH, "External_nist_test", "nist_smiles.txt")), torch.load(os.path.join(TEST_DATA_PATH, "External_nist_test", "nist_ir.pt"))
External_nist_test_loader = DataLoader(IRSmilesDataset(External_nist_test_ir, External_nist_test_smiles), batch_size=208, shuffle=False)
evaluate_retrieval(External_nist_test_loader, selected_features_nist, selected_smiles_nist, f"External NIST Test in {library_to_search_nist}")

In [ ]:
#@title 💊 Task 3: New Psychoactive Substances (NPS) Retrieval
#@markdown This task searches for NPS samples against specialized libraries of known and derivative psychoactive substances.
#@markdown ---
#@markdown ### Select a library to search against:
library_to_search_ps = "Derivative PS Library" #@param ["Existed PS Library", "Derivative PS Library", "Combined PS Library"]

# --- Map user selection to actual data ---
library_map_ps = {
    "Existed PS Library": (library_existed_ps_features, ps_smiles_existed),
    "Derivative PS Library": (library_derivative_ps_features, ps_smiles_derivative),
    "Combined PS Library": (library_combined_ps_features, ps_smiles_combined)
}
selected_features_ps, selected_smiles_ps = library_map_ps[library_to_search_ps]

# --- Load NPS test data and run ---
NPS_test_smiles, NPS_test_ir = load_smiles_list(os.path.join(TEST_DATA_PATH, "NPS", "NPS_smiles.txt")), torch.load(os.path.join(TEST_DATA_PATH, "NPS", "NPS_ir.pt"))
NPS_loader = DataLoader(IRSmilesDataset(NPS_test_ir, NPS_test_smiles), batch_size=208, shuffle=False)
evaluate_retrieval(NPS_loader, selected_features_ps, selected_smiles_ps, f"NPS Test in {library_to_search_ps}")

In [ ]:
#@title 📈 Task 4: Instant Spectra ID (IR Classification)
#@markdown Choose a test mode. You can either see the results on our pre-loaded test set or upload your own file to get an instant classification.
#@markdown ---
test_mode_ir = "See batch test results on our test set" #@param ["See batch test results on our test set", "Upload a single file for instant classification"]

# --- Define a new dataset class for IR Classification ---
class IRDataset(Dataset):
    def __init__(self, ir, labels): self.ir, self.labels = ir, labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return self.ir[i], self.labels[i]

if test_mode_ir == "See batch test results on our test set":
    print("--- Running batch test for IR Classifier ---")
    # Load data
    test_ir = torch.load(os.path.join(PS_TEST_DATA_PATH, "IR_Classifier", "test_ir.pt"))
    test_label = torch.load(os.path.join(PS_TEST_DATA_PATH, "IR_Classifier", "test_labels.pt"))
    test_loader = DataLoader(IRDataset(test_ir, test_label), batch_size=208, shuffle=False)

    # Run inference
    all_preds, all_labels = [], []
    with torch.no_grad():
        for ir_batch, label_batch in tqdm(test_loader, desc="IR Batch Test"):
            # Use the correct classifier encoder model
            ir_features = ir_classifier_encoder(ir_batch.to(device))
            logits = ir_classifier_model(ir_features)
            all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            all_labels.extend(label_batch.cpu().numpy())

    # Calculate and print confusion matrix
    label_0_pred_0 = sum(1 for p, l in zip(all_preds, all_labels) if l == 0 and p == 0)
    label_0_pred_1 = sum(1 for p, l in zip(all_preds, all_labels) if l == 0 and p == 1)
    label_1_pred_1 = sum(1 for p, l in zip(all_preds, all_labels) if l == 1 and p == 1)
    label_1_pred_0 = sum(1 for p, l in zip(all_preds, all_labels) if l == 1 and p == 0)
    print(f"\n--- Batch Test Results ---")
    print(f"Number of non-PS predicted as non-PS: {label_0_pred_0}")
    print(f"Number of non-PS predicted as PS: {label_0_pred_1}")
    print(f"Number of PS predicted as PS: {label_1_pred_1}")
    print(f"Number of PS predicted as non-PS: {label_1_pred_0}")

else: # Single file upload mode
    print("\n--- Running single file classification ---")

    # Define the prediction function specifically for this cell
    def predict_ir_classification(filepath, spectrum_type="absorbance spectrum"):
        print(f"Processing file: {os.path.basename(filepath)}")
        if filepath.lower().endswith('.csv'):
            # Assuming CSV has no header
            df = pd.read_csv(filepath, header=None)
            wavenumbers, transmittances = df.iloc[:, 0].values, df.iloc[:, 1].values
        elif filepath.lower().endswith('.jdx'):
            data = jcamp.jcamp_readfile(filepath)
            wavenumbers, transmittances = np.array(data['x']), np.array(data['y'])
        else:
            return "Error: Unsupported file format. Please upload a .csv or .jdx file."

        if spectrum_type != "absorbance spectrum": transmittances /= 100.0

        try:
            if wavenumbers[0] > 500: ir_data = preprocess_spectra_higer_500(wavenumbers, transmittances)
            else: ir_data = preprocess_spectra_lower_500(filepath, wavenumbers, transmittances)
        except Exception as e:
            return f"Error during preprocessing: {e}"

        ir_tensor = torch.tensor(ir_data, dtype=torch.float32).unsqueeze(0).to(device)[:,150:]

        with torch.no_grad():
            # Use the correct classifier models
            ir_feature = ir_classifier_encoder(ir_tensor)
            logits = ir_classifier_model(ir_feature)

        pred = torch.argmax(logits, dim=1).item()
        return "✅ Result: Psychoactive Substance" if pred == 1 else "✅ Result: Non-Psychoactive Substance"

    # File upload logic
    print("Please upload your spectra file:")
    uploaded = files.upload()

    if uploaded:
        for fn in uploaded.keys():
            print("-" * 30)
            # Create a temporary path to the uploaded file
            temp_path = os.path.join('/content/', fn)
            with open(temp_path, 'wb') as f:
                f.write(uploaded[fn])

            # Run prediction on the uploaded file
            # We assume absorbance for uploaded files, but this could be a dropdown too
            result = predict_ir_classification(temp_path, spectrum_type="absorbance spectrum")
            print(f"\n{result}")
            # Clean up the temporary file
            os.remove(temp_path)
    else:
        print("\nNo file uploaded. Please run the cell again and choose a file.")

In [ ]:
#@title ⚛️ Task 5: One-Click SMILES Analysis
#@markdown Choose a test mode. You can see the results on our pre-loaded test set or enter a single SMILES string for instant classification.
#@markdown ---
test_mode_smiles = "See batch test results on our test set" #@param ["See batch test results on our test set", "Enter a single SMILES string"]
smiles_to_predict = "CCO" #@param {type:"string"}

# --- Define a new dataset class for SMILES Classification ---
class SmilesDataset(Dataset):
    def __init__(self, smiles, labels): self.smiles, self.labels = smiles, labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return self.smiles[i], self.labels[i]

if test_mode_smiles == "See batch test results on our test set":
    print("--- Running batch test for SMILES Classifier ---")
    # Load data
    test_smiles_list = load_smiles_list(os.path.join(PS_TEST_DATA_PATH, "SMILES_Classifier", "test_smiles.txt"))
    test_label = torch.load(os.path.join(PS_TEST_DATA_PATH, "SMILES_Classifier", "test_labels.pt"))
    test_loader = DataLoader(SmilesDataset(test_smiles_list, test_label), batch_size=208, shuffle=False)

    # Run inference
    all_preds, all_labels = [], []
    with torch.no_grad():
        # Use the correct classifier encoder model
        tokenizer = smiles_classifier_encoder.smiles_tokenizer
        for smiles_batch, label_batch in tqdm(test_loader, desc="SMILES Batch Test"):
            encoded = [tokenizer.encode_plus(s, max_length=300, padding='max_length', truncation=True, return_tensors='pt') for s in smiles_batch]
            input_ids = torch.cat([item['input_ids'] for item in encoded], dim=0).to(device)
            attention_mask = torch.cat([item['attention_mask'] for item in encoded], dim=0).to(device)
            lengths = attention_mask.sum(dim=1)

            smiles_features = smiles_classifier_encoder.encode((input_ids, attention_mask), lengths)
            logits = smiles_classifier_model(smiles_features)

            all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            all_labels.extend(label_batch.cpu().numpy())

    # Calculate and print confusion matrix
    label_0_pred_0 = sum(1 for p, l in zip(all_preds, all_labels) if l == 0 and p == 0)
    label_0_pred_1 = sum(1 for p, l in zip(all_preds, all_labels) if l == 0 and p == 1)
    label_1_pred_1 = sum(1 for p, l in zip(all_preds, all_labels) if l == 1 and p == 1)
    label_1_pred_0 = sum(1 for p, l in zip(all_preds, all_labels) if l == 1 and p == 0)
    print(f"\n--- Batch Test Results ---")
    print(f"Number of non-PS predicted as non-PS: {label_0_pred_0}")
    print(f"Number of non-PS predicted as PS: {label_0_pred_1}")
    print(f"Number of PS predicted as PS: {label_1_pred_1}")
    print(f"Number of PS predicted as non-PS: {label_1_pred_0}")

else: # Single SMILES string mode
    print(f"\n--- Predicting for SMILES: '{smiles_to_predict}' ---")

    # Define the prediction function specifically for this cell
    def predict_smiles_classification(smiles):
        if not smiles or not smiles.strip():
            return "⚠️ Error: Please enter a valid SMILES string."

        # Use the normalize_smiles function we defined in the global init cell
        normalized = normalize_smiles(smiles.strip())
        print(f"Normalized SMILES: {normalized}")

        # Use the correct classifier models
        tokenizer = smiles_classifier_encoder.smiles_tokenizer
        encoded = tokenizer.encode_plus(text=normalized, max_length=300, padding='max_length', truncation=True, return_tensors='pt')

        input_ids = encoded['input_ids'].to(device)
        attention_mask = encoded['attention_mask'].to(device)
        lengths = attention_mask.sum(dim=1)

        with torch.no_grad():
            smiles_features = smiles_classifier_encoder.encode((input_ids, attention_mask), lengths)
            logits = smiles_classifier_model(smiles_features)

        pred = torch.argmax(logits, dim=1).item()
        return "✅ Result: Psychoactive Substance" if pred == 1 else "✅ Result: Non-Psychoactive Substance"

    # Run prediction based on user input from the form
    result = predict_smiles_classification(smiles_to_predict)
    print(f"\n{result}")